In [1]:
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import locale
import seaborn as sns
import matplotlib.pyplot as plt
import re
import pyproj
import folium

In [ ]:
mes = "jun"

In [ ]:
# Ruta de la carpeta que contiene los archivos de desglosados zonal
ruta_carpeta = 'Z:/01 base_datos/01 viajes_desglosados_FMS'

# Fechas de inicio y fin para el filtro
fecha_inicio = '20260601'
fecha_fin = '20260630'

# Lista para almacenar los DataFrames
dataframes = []

# Recorrer todos los archivos en la carpeta
for nombre_archivo in os.listdir(ruta_carpeta):
    if nombre_archivo.endswith('_viajedesglosado.csv'):
        # Extraer la fecha del nombre del archivo
        fecha_archivo = nombre_archivo[:8]  
        
        # Convertir la fecha a un formato adecuado para comparación
        fecha_archivo_dt = pd.to_datetime(fecha_archivo, format='%Y%m%d', errors='coerce')

        # Verificar si la fecha está dentro del rango deseado
        if fecha_inicio <= fecha_archivo <= fecha_fin:
            ruta_archivo = os.path.join(ruta_carpeta, nombre_archivo)
            # Leer el archivo CSV omitiendo la primera fila vacía
            df = pd.read_csv(ruta_archivo, encoding='latin', low_memory=False)
             # Eliminar filas que estén completamente vacías
            df = df.dropna(how='all')
            dataframes.append(df)

# Verificar si se encontraron DataFrames
if dataframes:
    # Consolidar todos los DataFrames en uno solo
    desg_zonal = pd.concat(dataframes, ignore_index=True)

    # # Guardar el DataFrame consolidado en un nuevo archivo
    # desg_zonal.to_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2024/Indicadores/Indicadores_python/desgl_zonal_nov24.csv', index=False)
else:
    print("No se encontraron archivos para consolidar.")
    
desg_zonal.head()

,Fecha,Concesión,Concesionario de Operación Planificado,Concesionario de Operación Real,ServViaje,Servicio,Orden Viaje,Id Viaje,Viaje Línea,Id Línea,...,DistNoRealizada,KmEjecutado,DespInicial,Puntualidad Preliminar,IdValPuntualidad,Cumplimiento Preliminar,FraHorCump,IdValCumplimiento,ICK Full Preliminar,ICK Preliminar
0,1/05/2026,ENGATIVA ZN,NaN,(105) GMOVIL ENGATIVA,20260501-AD0292021-1-12789,AD0292021,1,1,1,10292,...,0,37292,No,NaN,NaN,NaN,"3, 10",NaN,100.0,1.0
1,1/05/2026,ENGATIVA ZN,NaN,(105) GMOVIL ENGATIVA,20260501-AD0311008-1-12730,AD0311008,1,1,1,10311,...,0,36477,No,NaN,NaN,NaN,"3, 6",NaN,100.0,1.0
2,1/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260501-CE0650001-1-10533,CE0650001,1,2,1,10266,...,0,10904,Si,NaN,NaN,NaN,"3, 2",CDZ4,100.0,1.0
3,1/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260501-CE0650001-2-10533,CE0650001,2,3,2,10266,...,0,10904,No,NaN,NaN,NaN,"3, 2",CDZ4,100.0,1.0
4,1/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260501-CE0650001-3-10533,CE0650001,3,4,3,10266,...,0,10904,No,NaN,NaN,NaN,"3, 3",CDZ4,100.0,1.0


In [4]:
# Crear la tabla pivotada
tabla_pivot = desg_zonal.groupby(['Fecha', 'Vehículo Real ']).size().reset_index(name='Cantidad')
tabla_pivot = tabla_pivot.pivot(index='Vehículo Real ', columns='Fecha', values='Cantidad').fillna(0).astype(int)

# Agregar una fila de totales
tabla_pivot.loc['Total'] = tabla_pivot.sum()

# Verificar el resultado
tabla_pivot

Fecha,1/05/2026,10/05/2026,11/05/2026,12/05/2026,13/05/2026,14/05/2026,15/05/2026,16/05/2026,17/05/2026,18/05/2026,...,29/05/2026,3/05/2026,30/05/2026,31/05/2026,4/05/2026,5/05/2026,6/05/2026,7/05/2026,8/05/2026,9/05/2026
Vehículo Real,,,,,,,,,,,,,,,,,,,,,
Z50-2002,4,5,4,4,4,4,4,5,5,0,...,3,3,4,7,3,4,2,4,4,5
Z50-2003,2,5,4,4,3,3,3,2,5,4,...,3,0,3,3,5,4,4,4,3,2
Z50-2005,0,0,3,4,4,4,5,3,3,0,...,3,0,0,0,0,5,4,3,4,0
Z50-2007,5,3,5,2,4,1,4,0,0,0,...,3,6,3,0,3,4,2,5,4,3
Z50-2009,0,0,4,5,5,2,0,0,1,2,...,4,3,0,0,4,4,4,3,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Z52-4007,5,0,7,8,6,5,8,2,7,5,...,7,4,0,2,3,8,5,2,3,2
Z52-4008,0,0,0,0,0,0,2,4,0,0,...,4,0,5,0,0,0,0,0,0,0
Z52-4009,5,3,9,4,0,0,0,4,7,3,...,5,0,2,0,3,5,5,4,2,0


In [5]:
tabla_pivot.to_excel(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2023/Flota_Operativa_{mes}26_fms.xlsx')